# Решения: try/except CSV

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

In [ ]:
from pathlib import Path
import pandas as pd

def load_trips(path: Path) -> pd.DataFrame:
    try:
        return pd.read_csv(path)
    except FileNotFoundError as e:
        raise FileNotFoundError(
            f'Файл не найден: {path}. Положите trips.csv рядом с ноутбуком.'
        ) from e


def clean_trips(raw: pd.DataFrame) -> pd.DataFrame:
    return raw.dropna(subset=['distance_km', 'duration_min']).copy()


def assert_usable(frame: pd.DataFrame) -> None:
    if len(frame) == 0:
        raise ValueError('пустой DataFrame после очистки')
    for col in ('distance_km', 'duration_min'):
        if col not in frame.columns:
            raise ValueError(f'нет столбца {col}')


## Урок. 1. Успешная загрузка

In [ ]:
TRIPS_PATH = None
for p in (Path('trips.csv'), Path('../../data/trips.csv'), Path('../data/trips.csv')):
    if p.exists():
        TRIPS_PATH = p.resolve()
        break
assert TRIPS_PATH is not None
df = load_trips(TRIPS_PATH)
print(df.shape)

## Урок. 2. Ошибка пути

In [ ]:
caught = False
try:
    load_trips(Path('no_such_file.csv'))
except FileNotFoundError as e:
    caught = True
    print('Поймали:', e)
assert caught
ERROR_MSG_NOTE = (
    'Сообщение должно содержать путь и подсказку: '
    'положите trips.csv рядом с ноутбуком'
)
print(ERROR_MSG_NOTE)

## Урок. 3. Очистка

In [ ]:
raw = df.copy()
raw.loc[raw.index[0], 'duration_min'] = None
cleaned = clean_trips(raw)
assert len(cleaned) == len(raw) - 1
print(len(raw), len(cleaned))

## Урок. 4. assert_usable

In [ ]:
empty = pd.DataFrame(columns=['distance_km', 'duration_min'])
try:
    assert_usable(empty)
except ValueError as e:
    print('empty:', e)
bad = pd.DataFrame({'distance_km': [1.0, 2.0], 'hour': [8, 9]})
try:
    assert_usable(bad)
except ValueError as e:
    print('bad cols:', e)

## Урок. 5. Pipeline

In [ ]:
pipeline_df = clean_trips(load_trips(TRIPS_PATH))
assert_usable(pipeline_df)
print('pipeline', pipeline_df.shape)

## Урок. 6. Checkpoint

In [ ]:
WHY_TRY_FIRST = (
    'Ошибка файла должна быть понятной до fit: иначе traceback sklearn '
    'маскирует проблему данных.'
)
print(WHY_TRY_FIRST)

## ДЗ. 1–2. safe_load

In [ ]:
def safe_load(path):
    try:
        return pd.read_csv(path)
    except FileNotFoundError:
        return None


assert safe_load(Path('no_such_file.csv')) is None
ok = safe_load(TRIPS_PATH)
assert ok is not None and len(ok) > 0
print(ok.shape)

## ДЗ. 3. zone NaN

In [ ]:
sample = pd.DataFrame({
    'distance_km': [1.0, 2.0],
    'duration_min': [10.0, 20.0],
    'zone': [None, 'center'],
})
print(clean_trips(sample))

## ДЗ. 4. Пустой после clean

In [ ]:
tiny = pd.DataFrame({'distance_km': [1.0], 'duration_min': [None]})
cleaned = clean_trips(tiny)
status = 'empty after clean' if len(cleaned) == 0 else 'ok'
print(status)